In [ ]:
import pyspark.sql.functions as F
import requests

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
#Variables
CATALOG = "use1_prod_artemis_catalog_3718194974443840"  #change
SCHEMA = "tier1_raw"  #change

flights_table = f"{CATALOG}.{SCHEMA}.drone_mission_table"
ortho_table = f"{CATALOG}.{SCHEMA}.drone_ortho_table"

In [0]:
flights_df = spark.table(flights_table)
raw_ortho_df = spark.table(ortho_table)

if "ortho_exists" in raw_ortho_df.columns:
    finish_df = raw_ortho_df.filter(F.col("ortho_exists") == True)
else:

    finish_df = raw_ortho_df

missing_df = flights_df.join(
    finish_df,
    on=['site', 'trial', 'season', 'flight_date'],
    how='left_anti'
)

total_flights = flights_df.count()
total_finish = finish_df.count()
total_missing = missing_df.count()

print("-" * 40)
print(f" INVENTORY REPORT:")
print(f"Total flights detected: {total_flights}")
print(f"Orthomosaics physically generated: {total_finish}")
print(f"Flights pending processing: {total_missing}")
print("-" * 40)

if total_missing > 0:
    display(missing_df)
else:
    print(" Everything is up to date! There are no pending flights.")

----------------------------------------
 INVENTORY REPORT:
Total flights detected: 6
Orthomosaics already generated: 6
 Flights pending processing: 0
----------------------------------------
 Everything is up to date! There are no pending flights.


In [0]:
if total_missing > 0:
    print(f" Green flag: {total_missing} pending flights. Triggering ONE heavy job...")

    JOB_2_ID = 975722269459594  #change

    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host = ctx.apiUrl().get()
    token = ctx.apiToken().get()

    url = f"{host}/api/2.1/jobs/run-now"
    headers = {"Authorization": f"Bearer {token}"}

    # 1. We extract ALL the paths into a Python list
    flights_to_process = [row['flight_metadata_path'] for row in missing_df.select('flight_metadata_path').collect()]

    # 2. We convert that list into a single comma-separated String
    mixed_paths = ",".join(flights_to_process)

    # 3. We trigger a SINGLE "Run" by passing all the routes in the parameter
    data = {
        "job_id": JOB_2_ID,
        "notebook_params": {
            # We changed the parameter name to plural as a good practice
            "flight_metadata_paths": mixed_paths
        }
    }

    response = requests.post(url, headers=headers, json=data)

    if response.status_code == 200:
        run_id = response.json().get('run_id', 'Unknown')
        print(f" ->  Trigger successful! (Run ID: {run_id})")
        print(f" -> Were sent {len(flights_to_process)} missions to the heavy cluster in a single block.")
    else:
        print(f" ->  [ERROR] triggering Job: {response.text}")

else:
    print("  Red flag : There are no pending flights. Heavy cluster will remain off.")

 There are no pending flights..
